# EDA of SSVEP signals

## SSVEP signals


Steady-State Visual-Evoekd Potential is a brain resposne by a visual stumulus, flickering at a constant frequency between approximatively 6 an 100 Hz. The resposne manifests itself as an increace in amplitude of the stimulated frequency.

## Data

Based on our project specifications, the MAMEM dataset was selected for Explanatory Data Analysis (EDA). This dataset aligns perfectly with our requirements due to its limited number of classes, as we intend to use 4 to 6 flicking diodes corresponding to specific directional commands (left, right, up, down, straight, and back).

The MAMEM Steady-State Visually Evoked Potentials (SSVEP) database, which contains 256-channel EEG recordings from 11 subjects under flickering light stimulation, has been expanded with a second experimental dataset. During this experiment, subjects were exposed to non-overlapping flickering lights from five magenta boxes operating at frequencies of 6.66 Hz, 7.5 Hz, 8.57 Hz, 10 Hz, and 12 Hz, while 256-channel EEG data was captured.

## Libraries used

In [ ]:
# Importy podstawowych bibliotek do analizy EEG i wizualizacji danych.
# moabb dostarcza dataset MAMEM3 oraz paradygmat SSVEP.
# scipy.signal.welch służy do estymacji mocy widmowej (PSD).
# sklearn.cross_decomposition.CCA daje klasyczny klasyfikator dla SSVEP.
# scipy.signal.cheby1 i filtfilt są używane do wstępnej filtracji 5-45 Hz.
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from moabb.datasets import MAMEM3
from moabb.paradigms import SSVEP
from scipy.signal import welch
from sklearn.cross_decomposition import CCA
from scipy.signal import cheby1, filtfilt


In [ ]:
# Tworzymy instancję datasetu oraz obiekt paradygmatu.
# Dataset MAMEM3 zawiera nagrania EEG z eksperymentów SSVEP.
# Paradigma SSVEP określa sposób pobierania i organizacji danych dla klasyfikacji.
dataset = MAMEM3()
paradigm = SSVEP()

Choosing the first None classes from all possible events


Data from first subject

In [ ]:
# Pobieramy dane dla pierwszego użytkownika z datasetu.
# X zawiera triale EEG w formacie (trial, kanał, próbki).
# labels zawiera etykiety częstotliwości stymulacji.
# metadata przechowuje dodatkowe informacje o każdym trialu.
X, labels, metadata = paradigm.get_data(
    dataset=dataset,
    subjects=[1]
)

fs = 128

# MAMEM3 korzysta z 14-kanałowego układu EPOC. Dla SSVEP interesują nas przede wszystkim
# kanały potyliczne O1 i O2; w tym układzie nie ma Oz ani POz.
EPOC_CHANNELS = [
    "AF3", "F7", "F3", "FC5", "T7", "P7", "O1",
    "O2", "P8", "T8", "FC6", "F4", "F8", "AF4"
]
OCCIPITAL_CHANNELS = ["O1", "O2"]
OCCIPITAL_CHANNELS_INDICES = [
    EPOC_CHANNELS.index(channel) for channel in OCCIPITAL_CHANNELS
]


def cheby_bandpass_filter_signal(
    signal: np.ndarray,
    lowcut: float = 5.0,
    highcut: float = 45.0,
    fs: int = 128,
    order: int = 4,
    ripple_db: float = 0.5,
) -> np.ndarray:
    """Band-pass filters a single EEG channel with a zero-phase Chebyshev type I filter."""
    nyquist = 0.5 * fs
    b, a = cheby1(
        order,
        ripple_db,
        [lowcut / nyquist, highcut / nyquist],
        btype="bandpass",
    )
    return filtfilt(b, a, signal)


def filter_eeg_trials(
    trials: np.ndarray,
    lowcut: float = 5.0,
    highcut: float = 45.0,
    fs: int = 128,
) -> np.ndarray:
    """Applies the same band-pass filter independently to every trial and EEG channel."""
    filtered = np.empty_like(trials, dtype=float)

    for trial_idx, trial in enumerate(trials):
        for channel_idx in range(trial.shape[0]):
            filtered[trial_idx, channel_idx] = cheby_bandpass_filter_signal(
                trial[channel_idx],
                lowcut=lowcut,
                highcut=highcut,
                fs=fs,
            )

    return filtered


# Wstępnie filtrujemy całe EEG do pasma 5-45 Hz. Dzięki temu PSD, SNR i CCA nie pracują
# na bardzo wolnym dryfcie ani wysokoczęstotliwościowym szumie.
X_filtered = filter_eeg_trials(X, lowcut=5.0, highcut=45.0, fs=fs)

# Osobny widok tylko na O1/O2 przyda się później w SNR, CCA i FBCCA.
X_occipital = X_filtered[:, OCCIPITAL_CHANNELS_INDICES, :]


### Basic dataset features

In [ ]:
# Wypisujemy podstawowe informacje o zbiorze danych.
# Dzięki temu sprawdzamy jego kształt, liczbę triali i rozkład klas.
print(X.shape)
print(labels.shape)
print(labels)
print(metadata.head())
print(pd.Series(labels).value_counts())

As we can see, the trial counts for each frequency are similar, ranging from 20 to 30.

For SSVEP signals, we are most interested in occipital brain signals. In the 14-channel EPOC montage used here, the available occipital electrodes are O1 and O2; Oz and POz are not present.

![Occipital Lobe](assets/Occipital.jpg)


Simple data representations

In this part, I used basic visual representations of the EEG signal to understand its structure before applying classification methods. One of the most important tools was the Welch function, which estimates the Power Spectral Density (PSD) of the signal.

Power Spectral Density shows how the power of the signal is distributed across different frequencies. In the context of SSVEP signals, this is especially important because the brain response should contain stronger activity at the frequency of the visual stimulus, for example 6.66 Hz, and sometimes also at its harmonics, such as 13.32 Hz or 19.98 Hz.

The Welch method works by dividing the signal into shorter overlapping segments, calculating the spectrum for each segment, and then averaging the results. This makes the PSD estimation more stable and less noisy than calculating a single Fourier transform on the whole signal.

https://en.wikipedia.org/wiki/Welch%27s_method

In [ ]:
# Wybieramy pojedynczy trial i analizujemy tylko kanały potyliczne O1/O2.
# W poprzedniej wersji indeks trialu został przypadkowo użyty także jako indeks kanału,
# przez co PSD było liczone dla kanału niepotylicznego.
trial_idx = 8
trial = X_filtered[trial_idx]

current_label = labels[trial_idx]

target_freqs = [6.66, 7.50, 8.57, 10.00, 12.00]

# Liczymy PSD osobno dla O1 i O2, a następnie uśredniamy moce widmowe.
# Nie uśredniamy sygnałów w czasie, żeby nie osłabić odpowiedzi przez różnice fazowe.
occipital_psds = []

for channel_idx in OCCIPITAL_CHANNELS_INDICES:
    freqs, channel_psd = welch(
        trial[channel_idx],
        fs=fs,
        nperseg=fs * 2,
    )
    occipital_psds.append(channel_psd)

psd = np.mean(occipital_psds, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(freqs, psd, color='black', linewidth=1)

# Rysujemy pionowe linie na częstotliwościach stymulacji oraz ich harmonicznych.
for f in target_freqs:
    plt.axvline(x=f, color='red', linestyle='--', alpha=0.6)
    plt.text(f, plt.ylim()[1] * 0.9, f'{f}Hz', rotation=90, color='red', fontsize=8, ha='right')

    h2 = f * 2
    if h2 <= 30:
        plt.axvline(x=h2, color='orange', linestyle=':', alpha=0.4)
        plt.text(h2, plt.ylim()[1] * 0.8, f'2x{f}Hz', rotation=90, color='orange', fontsize=7, ha='right')

plt.title(
    f"Trial Index: {trial_idx} | O1/O2 | Subject was looking at: {current_label} Hz",
    fontsize=14,
    fontweight='bold',
    pad=20,
)

plt.xlim(0, 30)
plt.xlabel('Frequency (Hz)')
plt.ylabel('Power Spectral Density')
plt.show()


Despite the application of Welch’s power spectral density (PSD) estimation, the observed frequency components exhibited insufficient alignment with the target stimulus frequencies. The presence of spurious local maxima and the lack of distinct peaks at the expected harmonics suggest a low signal-to-noise ratio (SNR). Consequently, alternative feature extraction methods may be required to isolate the SSVEP response from background cortical activity.To ilustrate this problem I prepared diagram below.

In [ ]:
# Porównujemy średnie PSD dla każdej klasy na kanałach O1/O2.
# Dzięki temu nie rozmywamy odpowiedzi SSVEP przez uśrednianie kanałów z całej głowy.
unique_labels = np.unique(labels)

for lab in unique_labels:
    X_lab = X_filtered[labels == lab]

    psds = []
    for trial in X_lab:
        trial_occipital_psds = []

        for channel_idx in OCCIPITAL_CHANNELS_INDICES:
            freqs, psd = welch(
                trial[channel_idx],
                fs=fs,
                nperseg=fs * 2,
            )
            trial_occipital_psds.append(psd)

        psds.append(np.mean(trial_occipital_psds, axis=0))

    mean_psd = np.mean(psds, axis=0)
    plt.plot(freqs, mean_psd, label=str(lab))

plt.xlim(0, 40)
plt.xlabel("Częstotliwość [Hz]")
plt.ylabel("PSD")
plt.title("Średnie PSD dla klas SSVEP - O1/O2, bandpass 5-45 Hz")
plt.legend()
plt.show()


As illustrated in plot above the grand average power spectral density (PSD) curves fail to show distinct, class-specific peaks at their respective target stimulation frequencies (6.66 Hz, 7.50 Hz, 8.57 Hz, 10.00 Hz, and 12.00 Hz). Instead, the spectral profiles across all experimental conditions are dominated by a broad, overlapping elevation in power within the 7–14 Hz band. This pronounced morphology is indicative of strong endogenous alpha rhythms, which effectively mask the narrower, stimulus-entrained SSVEP components and yield an insufficient signal-to-noise ratio (SNR) for univariate frequency-domain classification.

To address the limitations of raw spectral decomposition via Welch’s method, a Signal-to-Noise Ratio (SNR) metric was employed to quantify the magnitude of the SSVEP response relative to the baseline noise floor. Following power spectral density (PSD) estimation, the SNR was calculated as the ratio of the spectral power at the target stimulation frequency (ftarget​) to the mean power of the adjacent frequency bins. In the context of SSVEP decoding, this approach provides an objective, normalized index of stimulus-entrained cortical activity, effectively isolating narrow-band evoked responses from broadband background electroencephalographic (EEG) noise and mitigating individual variations in baseline alpha activity.

In [ ]:
# Poniżej definiujemy funkcje do liczenia SNR dla konkretnej częstotliwości.
# Idea jest prosta: porównujemy moc sygnału na częstotliwości docelowej z mocą tła w okolicznych binach.
# Jeżeli odpowiedź SSVEP jest silna, SNR powinien rosnąć na częstotliwości stymulacji.


def get_nearest_freq_idx(freqs: np.ndarray,
                        target_freq: float) -> int:
    """ Returns the index of the nearest frequency bin.
    
    Args:
        freqs (array_like): Array or list of frequency bins (ints or floats).
        target_freq (float): Target frequency to find.
    
    Returns: 
        int: Index of the nearest frequency bin.

    Raises:
        ValueError: If target_freq is negative or if any frequency in freqs is negative.
        ValueError: If the frequency array is empty.
        TypeError: If target_freq is not a number or freqs contains non-numeric types.
    """

    # Zamieniamy dane na tablicę numpy, żeby łatwiej operować na nich w sposób wektorowy.
    freqs = np.asanyarray(freqs)

    # Sprawdzamy, czy wejście jest sensowne i nieujemne.
    if target_freq < 0 or np.any(freqs < 0):
        raise ValueError("Frequencies must be non-negative.")
    
    if freqs.size == 0:
        raise ValueError("Frequency array cannot be empty.")

    if not np.issubdtype(freqs.dtype, np.number):
        raise ValueError("freqs must contain only integers or floats.")

    # Zwracamy indeks najbliższego binu częstotliwości do target_freq.
    return np.argmin(np.abs(freqs - target_freq))


def compute_snr_at_freq(freqs: np.ndarray,
                        psd: np.ndarray,
                        target_freq: float, 
                        noise_width: int = 2, 
                        skip_width: int = 1) -> tuple[float, float]:
    """Computes SNR for a single frequency.

    Args:
        freqs (array_like): Array or list of frequency bins (ints or floats).
        target_freq (float): Target frequency to find.
        psd (array): Power spectral density for a single EEG signal. 
        noise_width  (int): Number of bins to take as noise on each side of the target frequency.
        skip_width (int): Number of adjacent bins to skip on each side of the target frequency.

    Returns
        snr (float) : SNR as the ratio of signal power to noise power. 
        snr_db (float) SNR in decibels
    
    """

    # Sprawdzamy, czy parametry są sensowne.
    if(noise_width < 1 or skip_width < 0):
        raise ValueError("noise_width must be >= 1 and skip_width must be >= 0.")
    
    if len(freqs) != len(psd):
        raise ValueError("Length of freqs and psd must be the same.")
    
    if target_freq < 0:
        raise ValueError("target_freq must be non-negative.")

    # Znajdujemy indeks najbliższego binu dla docelowej częstotliwości.
    idx = get_nearest_freq_idx(freqs, target_freq)

    # Moc sygnału w docelowym binie.
    signal_power = psd[idx]

    # Definiujemy zakres binów tła po lewej i prawej stronie docelowego binu.
    left_start = idx - skip_width - noise_width
    left_end = idx - skip_width

    right_start = idx + skip_width + 1
    right_end = idx + skip_width + noise_width + 1

    # Jeśli zakres wychodzi poza tablicę, zwracamy NaN, bo nie da się policzyć sensownego SNR.
    if left_start < 0 or right_end > len(psd):
        return np.nan, np.nan

    # Łączymy bity po lewej i prawej stronie, aby uzyskać szum tła.
    noise_bins = np.concatenate([
        psd[left_start:left_end],
        psd[right_start:right_end]
    ])

    # Średnia moc tła stanowi reprezentację poziomu szumu.
    noise_power = np.mean(noise_bins)

    # SNR jako stosunek mocy sygnału do mocy tła.
    snr = signal_power / noise_power
    # Zamieniamy na decybele dla wygodniejszej interpretacji.
    snr_db = 10 * np.log10(snr)

    return snr, snr_db

In [ ]:
# Tworzymy tabelę z wynikami dla każdej próby (trial) i każdej częstotliwości docelowej.
# SNR liczymy tylko na O1/O2, zamiast uśredniać po wszystkich kanałach EEG.
rows = []

for trial_idx, trial in enumerate(X_filtered):
    true_label = float(labels[trial_idx])
    trial_result = {"trial": trial_idx, "true_label": true_label}

    # PSD liczymy wyłącznie dla kanałów potylicznych, gdzie odpowiedź SSVEP jest najsilniejsza.
    channel_psds = []
    for channel_idx in OCCIPITAL_CHANNELS_INDICES:
        f_bins, psd_vals = welch(
            trial[channel_idx],
            fs=fs,
            nperseg=trial.shape[1],
        )
        channel_psds.append((f_bins, psd_vals))

    # Dla każdej częstotliwości docelowej liczymy SNR po O1/O2.
    for target_freq in target_freqs:
        snrs_for_channels = []
        snrs_db_for_channels = []

        for f_bins, psd_vals in channel_psds:
            snr, snr_db = compute_snr_at_freq(
                freqs=f_bins,
                psd=psd_vals,
                target_freq=target_freq,
                noise_width=2,
                skip_width=1,
            )
            snrs_for_channels.append(snr)
            snrs_db_for_channels.append(snr_db)

        trial_result[f"SNR_{target_freq:.2f}"] = np.nanmean(snrs_for_channels)
        trial_result[f"SNRdB_{target_freq:.2f}"] = np.nanmean(snrs_db_for_channels)

    rows.append(trial_result)

snr_df = pd.DataFrame(rows)

# Wybieramy częstotliwość z największym SNR.
best_col = snr_df[[f"SNR_{f:.2f}" for f in target_freqs]].idxmax(axis=1)
snr_df["predicted_label"] = best_col.str.replace("SNR_", "").astype(float)

snr_df["correct"] = snr_df["true_label"] == snr_df["predicted_label"]


In [ ]:
# Liczymy średnią accuracy dla klasyfikatora opartego na SNR.
# To daje nam prosty punkt odniesienia przed przejściem do bardziej zaawansowanego CCA.
accuracy = snr_df["correct"].mean()
print(f"Accuracy: {accuracy:.2%}")

Restricting the SNR calculation to the occipital O1/O2 channels and applying a 5–45 Hz band-pass filter provides a cleaner baseline than averaging across all EEG channels. The exact accuracy is printed above after rerunning the notebook. Because an SNR-only classifier still uses a relatively simple univariate spectral criterion, the next step is Canonical Correlation Analysis (CCA), which can exploit the multichannel structure of the occipital EEG and harmonic reference signals.


In [ ]:
# Ta komórka generuje referencyjne sygnały sinus/cosinus dla każdego możliwego target_freq.
# W klasycznym CCA porównujemy EEG z tymi wzorcami, bo SSVEP powinien korelować z takimi falami.

def generate_reference_signals(freq, length_sec, fs, n_harmonics=3):
    """Generates sine and cosine reference signals for CCA."""
    # Wektor czasu jest potrzebny do wygenerowania sygnałów o odpowiedniej długości.
    t = np.linspace(0, length_sec, int(fs * length_sec), endpoint=False)
    refs = []
    # Dla każdej harmonicznej tworzymy sinus i cosinus. To pozwala modelowi uchwycić zarówno fazę, jak i zmianę w czasie.
    for h in range(1, n_harmonics + 1):
        refs.append(np.sin(2 * np.pi * h * freq * t))
        refs.append(np.cos(2 * np.pi * h * freq * t))
    return np.array(refs) # Shape: (2 * n_harmonics, samples)

## CCA 

In [ ]:
# Definiujemy funkcję liczącą korelację CCA pomiędzy pojedynczym trialem EEG a wzorcem referencyjnym.
# CCA jest używane, bo może znaleźć wspólne składowe między wielokanałowym EEG a wzorcem sygnału.

def get_cca_correlation(data : np.ndarray, ref : np.ndarray) -> float:
    """Computes the maximum canonical correlation between EEG data and reference.
    
        Args:
            data: (channels, samples) EEG data for a single trial.
            ref: (n_refs, samples) Reference signals for a specific frequency (including harmonics

        Returns:
            corr: Maximum canonical correlation coefficient.

    """

    # CCA działa na danych w formie (samples, variables), więc transponujemy wejście EEG.
    X = data.T 
    Y = ref.T
    
    # Ustawiamy 1 komponent. W naszym przypadku liczymy jedną największą korelację.
    cca = CCA(n_components=1)
    X_c, Y_c = cca.fit_transform(X, Y)
    
    # Korelacja między składowymi uzyskanymi przez CCA daje miarę dopasowania do wzorca.
    corr = np.corrcoef(X_c[:, 0], Y_c[:, 0])[0, 1]
    return corr

# Druga funkcja do generowania referencyjnych sygnałów, tym razem z bardziej czytelną typizacją.
def generate_reference_signals(freq: float, length_sec: float, fs: int, n_harmonics: int = 3) -> np.ndarray:
    """Generates sine/cosine reference signals for a given stimulus frequency.

    Args:
        freq (float): Target stimulation frequency in Hz.
        length_sec (float): Trial duration in seconds.
        fs (int): Sampling rate in Hz.
        n_harmonics (int): Number of harmonics to include.

    Returns:
        np.ndarray: Reference signals with shape (2 * n_harmonics, n_samples).
    """
    # Przygotowujemy wektor czasu na podstawie częstotliwości próbkowania.
    n_samples = int(fs * length_sec)
    t = np.arange(n_samples) / fs
    refs = []

    # Dla każdej harmonicznej h dodajemy sinus i cosinus.
    for h in range(1, n_harmonics + 1):
        refs.append(np.sin(2 * np.pi * h * freq * t))
        refs.append(np.cos(2 * np.pi * h * freq * t))

    return np.vstack(refs)

In [ ]:
# W tej komórce uruchamiamy klasyczny CCA dla każdego trialu.
# Używamy już przefiltrowanych kanałów O1/O2 zamiast całego nieprzefiltrowanego EEG.
length_sec = X_occipital.shape[2] / fs

# Tworzymy słownik referencji dla każdej z docelowych częstotliwości.
references = {
    f: generate_reference_signals(f, length_sec, fs, n_harmonics=3)
    for f in target_freqs
}

results = []

for trial_idx, trial in enumerate(X_occipital):
    correlations = []

    for f in target_freqs:
        ref = references[f]
        rho = get_cca_correlation(trial, ref)
        correlations.append(rho)

    predicted_idx = np.argmax(correlations)
    predicted_freq = target_freqs[predicted_idx]

    results.append({
        "trial": trial_idx,
        "true_label": labels[trial_idx],
        "predicted_label": predicted_freq,
    })

cca_df = pd.DataFrame(results)
cca_df["correct"] = np.isclose(
    cca_df["true_label"].astype(float),
    cca_df["predicted_label"].astype(float),
    atol=0.01,
)

print(f"CCA Accuracy: {cca_df['correct'].mean():.2%}")


After restricting the analysis to O1/O2 and applying the 5–45 Hz band-pass filter, the CCA result above becomes the relevant baseline. Standard CCA already uses sine/cosine references for the fundamental stimulation frequency and its harmonics, but it treats the analysed spectrum as a single band. The next step is Filter Bank CCA (FBCCA), where several filtered subbands are scored separately and then combined with decreasing weights. This usually gives more emphasis to the frequency ranges carrying the fundamental and harmonic SSVEP components while reducing broadband interference.


Next logical step after CCA is to use FBCCA

In [ ]:
from scipy.signal import butter

# FBCCA używa kilku filtrów pasmowych, aby osobno ocenić informacje z kolejnych
# zakresów obejmujących częstotliwość podstawową i harmoniczne SSVEP.


def generate_reference_signals(
    freq: float,
    length_sec: float,
    fs: int,
    n_harmonics: int = 3,
) -> np.ndarray:
    """Generates sine/cosine reference signals for a given stimulus frequency."""
    n_samples = int(fs * length_sec)
    t = np.arange(n_samples) / fs
    refs = []

    for h in range(1, n_harmonics + 1):
        refs.append(np.sin(2 * np.pi * h * freq * t))
        refs.append(np.cos(2 * np.pi * h * freq * t))

    return np.vstack(refs)


def bandpass_filter_signal(
    signal: np.ndarray,
    lowcut: float,
    highcut: float,
    fs: int,
    order: int = 4,
) -> np.ndarray:
    """Band-pass filters a single EEG channel for one FBCCA subband."""
    nyq = 0.5 * fs
    b, a = butter(
        order,
        [lowcut / nyq, highcut / nyq],
        btype="bandpass",
    )
    return filtfilt(b, a, signal)


def get_cca_correlation(data: np.ndarray, ref: np.ndarray) -> float:
    """Computes the maximum canonical correlation between EEG data and a reference."""
    X = data.T
    Y = ref.T

    cca = CCA(n_components=1)
    X_c, Y_c = cca.fit_transform(X, Y)

    corr = np.corrcoef(X_c[:, 0], Y_c[:, 0])[0, 1]
    return corr


def fbcca_predict(
    trials: np.ndarray,
    true_labels: np.ndarray,
    target_freqs: list[float],
    fs: int,
    filter_bands: list[tuple[float, float]] | None = None,
):
    # W klasycznym FBCCA subbandy są nakładające się: kolejne filtry podnoszą dolną
    # granicę pasma, ale zachowują wspólną górną granicę. Dzięki temu kolejne banki
    # kładą coraz większy nacisk na harmoniczne.
    if filter_bands is None:
        filter_bands = [(6, 45), (14, 45), (22, 45)]

    length_sec = trials.shape[2] / fs
    references = {
        f: generate_reference_signals(f, length_sec, fs, n_harmonics=3)
        for f in target_freqs
    }

    # Standardowy typ malejących wag używany w FBCCA:
    # w_m = m^(-1.25) + 0.25, gdzie m zaczyna się od 1.
    weights = np.array([
        (band_idx + 1) ** (-1.25) + 0.25
        for band_idx in range(len(filter_bands))
    ])

    results = []

    for trial_idx, trial in enumerate(trials):
        scores = []

        for target_freq in target_freqs:
            subband_scores = []

            for lowcut, highcut in filter_bands:
                filtered_trial = np.zeros_like(trial, dtype=float)

                for ch_idx in range(trial.shape[0]):
                    filtered_trial[ch_idx] = bandpass_filter_signal(
                        trial[ch_idx],
                        lowcut,
                        highcut,
                        fs,
                    )

                rho = get_cca_correlation(
                    filtered_trial,
                    references[target_freq],
                )

                # W FBCCA korelacja jest zwykle podnoszona do kwadratu przed agregacją.
                subband_scores.append(rho ** 2)

            score = np.dot(
                np.array(subband_scores),
                weights[:len(subband_scores)],
            )
            scores.append(score)

        pred_idx = int(np.argmax(scores))
        predicted_freq = float(target_freqs[pred_idx])

        results.append({
            "trial": trial_idx,
            "true_label": float(true_labels[trial_idx]),
            "predicted_label": predicted_freq,
        })

    fbcca_df = pd.DataFrame(results)
    fbcca_df["correct"] = np.isclose(
        fbcca_df["true_label"].astype(float),
        fbcca_df["predicted_label"].astype(float),
        atol=0.01,
    )
    return fbcca_df


# FBCCA uruchamiamy na tych samych przefiltrowanych kanałach O1/O2 co zwykłe CCA,
# dzięki czemu porównanie obu metod jest uczciwe.
fbcca_df = fbcca_predict(
    X_occipital,
    labels,
    target_freqs,
    fs,
)

print(f"FBCCA Accuracy: {fbcca_df['correct'].mean():.2%}")
fbcca_df.head()


In [ ]:
print('OCCIPITAL_CHANNELS:', OCCIPITAL_CHANNELS)
print('OCCIPITAL_CHANNELS_INDICES:', OCCIPITAL_CHANNELS_INDICES)
print('X_occipital shape:', X_occipital.shape)


In [ ]:
# Ta komórka służy do porównania kilku wariantów FBCCA.
# Każdy wariant różni się zestawem subbandów i wagami.


def evaluate_fbcca_variant(filter_bands, weights):
    results = []

    for trial_idx, trial in enumerate(X_occipital):
        scores = []

        for target_freq in target_freqs:
            subband_scores = []

            for lowcut, highcut in filter_bands:
                filtered_trial = np.zeros_like(trial, dtype=float)

                for ch_idx in range(trial.shape[0]):
                    filtered_trial[ch_idx] = bandpass_filter_signal(
                        trial[ch_idx],
                        lowcut,
                        highcut,
                        fs,
                    )

                rho = get_cca_correlation(
                    filtered_trial,
                    references[target_freq],
                )
                subband_scores.append(rho ** 2)

            score = np.dot(
                np.array(subband_scores),
                np.array(weights[:len(subband_scores)]),
            )
            scores.append(score)

        pred_idx = int(np.argmax(scores))
        predicted_freq = float(target_freqs[pred_idx])

        results.append({
            "trial": trial_idx,
            "true_label": float(labels[trial_idx]),
            "predicted_label": predicted_freq,
        })

    fbcca_df = pd.DataFrame(results)
    fbcca_df["correct"] = np.isclose(
        fbcca_df["true_label"].astype(float),
        fbcca_df["predicted_label"].astype(float),
        atol=0.01,
    )
    return fbcca_df["correct"].mean()


# Kandydaci zachowują ideę filter banku: kolejne pasma mają coraz wyższy low-cut,
# a górna granica pozostaje na 45 Hz, żeby zachować harmoniczne.
standard_weights = [
    1 ** (-1.25) + 0.25,
    2 ** (-1.25) + 0.25,
    3 ** (-1.25) + 0.25,
]

candidates = [
    ([(6, 45), (14, 45), (22, 45)], standard_weights),
    ([(5, 45), (12, 45), (20, 45)], standard_weights),
    ([(6, 45), (12, 45), (18, 45)], standard_weights),
    ([(6, 45), (16, 45), (26, 45)], standard_weights),
    ([(6, 45), (14, 45)], standard_weights[:2]),
]

for bands, weights in candidates:
    acc = evaluate_fbcca_variant(bands, weights)
    print(bands, [round(w, 3) for w in weights], f"-> {acc:.2%}")
